<a href="https://colab.research.google.com/github/prroud/AI_learning_journey/blob/main/Intel_Image_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [53]:
import kagglehub

path = kagglehub.dataset_download("puneet6060/intel-image-classification")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'intel-image-classification' dataset.
Path to dataset files: /kaggle/input/intel-image-classification


In [54]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
import os
import copy
from torchmetrics.classification import Accuracy

In [55]:
!pip install torchmetrics

In [56]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [57]:
data_dir = path

full_training_dataset = ImageFolder(root=f"{path}/seg_train/seg_train", transform=train_transform)
full_val_dataset = ImageFolder(root=f"{path}/seg_test/seg_test", transform=val_transform)

train_dataloader = DataLoader(dataset=full_training_dataset, batch_size=32, shuffle=True, num_workers=2)
val_dataloader = DataLoader(dataset=full_val_dataset, batch_size=32, shuffle=False, num_workers=2)

In [58]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [59]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False

num_features = model.fc.in_features
model.fc = nn.Linear(in_features=num_features, out_features=6)
model = model.to(device)
model


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [60]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.fc.parameters(), lr=1e-3)
accuracy = Accuracy(task="multiclass", num_classes=6).to(device)

In [61]:
def train_model(model, criterion, optimizer, num_epochs=5):
    dataloaders = {
        "train": train_dataloader,
        "val": val_dataloader
    }

    datasets_lengths = {
        "train": len(train_dataloader.dataset),
        "val": len(val_dataloader.dataset)
    }

    best_weights = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1}/{num_epochs}")

        for phase in ["train", "val"]:
            if phase == "train":
                model.train()
            if phase == "val":
                model.eval()

            running_loss = 0.0
            accuracy.reset()

            for X_batch, y_batch in dataloaders[phase]:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)

                with torch.set_grad_enabled(phase=="train"):
                    y_preds = model(X_batch)
                    loss = criterion(y_preds, y_batch)

                    if phase == "train":
                        optimizer.zero_grad()
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * X_batch.size(0)
                accuracy.update(y_preds, y_batch)

            epoch_loss = running_loss / datasets_lengths[phase]
            epoch_acc = accuracy.compute().item()

            print(f'{phase} loss: {epoch_loss:.4f}, accuracy: {epoch_acc:.2f}')

            if phase == "val" and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_weights = copy.deepcopy(model.state_dict())
        print('\n')

    model.load_state_dict(best_weights)
    return model




In [ ]:
model = train_model(model=model, criterion=criterion, optimizer=optimizer, num_epochs=5)

Epoch 1/5


In [ ]:
for name, child in model.named_children():
    if name in ["layer3", "layer4"]:
        for param in child.parameters():
            param.requires_grad = True

optimizer_fine = torch.optim.Adam([
    {"params": model.layer3.parameters(), "lr": 1e-5},
    {"params": model.layer4.parameters(), "lr": 1e-5},
    {"params": model.fc.parameters(), "lr": 1e-4}
])

model = train_model(model=model, criterion=criterion, optimizer=optimizer_fine, num_epochs=5)